# Data: PubMed 20k RCT

Downloads the dataset from the authors' GitHub at runtime (it is not redistributed here), saves train/dev/test, and fixes a balanced 1,000-sentence evaluation set used by every model. CPU runtime.

In [ ]:
# ===== STEP 1: download PubMed 20k RCT, explore it, and fix the evaluation sets =====
import os, urllib.request
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/sci-classify/data"
os.makedirs(OUT, exist_ok=True)
BASE = "https://raw.githubusercontent.com/Franck-Dernoncourt/pubmed-rct/master/PubMed_20k_RCT/"

def load(split):
    """Each abstract starts with '###<id>'; each sentence line is 'LABEL<TAB>text'; abstracts end with a blank line."""
    raw = urllib.request.urlopen(BASE + f"{split}.txt").read().decode("utf-8").splitlines()
    rows, abstract, lines = [], None, []
    for line in raw + [""]:
        if line.startswith("###"):
            abstract, lines = line[3:], []
        elif line.strip() == "":
            for i, (label, text) in enumerate(lines):
                rows.append({"abstract_id": abstract, "line_number": i, "total_lines": len(lines),
                             "label": label, "text": text})
            lines = []
        else:
            label, text = line.split("\t", 1)
            lines.append((label, text))
    return pd.DataFrame(rows)

data = {s: load(s) for s in ("train", "dev", "test")}
for s, df in data.items():
    print(f"{s:5s}: {df.abstract_id.nunique():>6,} abstracts, {len(df):>7,} sentences")
    df.to_parquet(f"{OUT}/{s}.parquet", index=False)

print("\nLabel distribution in the test set:")
print((data["test"].label.value_counts(normalize=True) * 100).round(1).to_string())

print("\nOne example abstract:")
ex = data["test"][data["test"].abstract_id == data["test"].abstract_id.iloc[0]]
for r in ex.itertuples():
    print(f"  [{r.label:11s}] {r.text[:110]}")

# Fixed evaluation sets, drawn once with a seed and reused by every model:
# - 'test_llm': 1,000 sentences, stratified by label, small enough to send to an LLM API cheaply
# - the full test set is used for the models that run locally
test = data["test"]
test_llm = test.groupby("label").sample(n=200, random_state=42).reset_index(drop=True)
test_llm.to_parquet(f"{OUT}/test_llm_1000.parquet", index=False)
print(f"\nSaved a fixed, balanced 1,000-sentence LLM test set ({test_llm.label.value_counts().min()} per label).")
print("Files saved to", OUT)
